# PepXPro ML Benchmark

**Question:** Do PepXPro curation choices (redundancy removal + Kennard–Stone sampling)
improve Random Forest generalisation on unseen peptide–protein complexes?

**Two scenarios:**

- **Scenario A — same-pool 80/20:** curate N entries from the full dataset using each
  strategy, then split 80/20; train on 80%, test on 20%.
  *Question: does curation improve within-set generalisation?*

- **Scenario B — partial-helix held out:** entries with `peptide_ss_class = "partial-helix"`
  (n≈96) are held out as a fixed test set; each strategy curates the remaining entries for
  training.
  *Question: does curation improve out-of-class generalisation?*

**12 structural features** (all derived from PepXPro pipeline): `pep_len`, `rec_len`,
`peptide_pct_helix/beta/coil`, `receptor_pct_helix/beta/coil`,
`n_contacts` (4.5 Å), `n_hbonds` (N/O/S proxy, 3.5 Å), `binding_kd`, `resolution`.

**Metrics:** RMSE, MAE, Pearson r, Spearman ρ.

## 1. Setup

In [ ]:
import sys, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ── PepXPro gen_sample modules ───────────────────────────────────────────────
REPO = Path("../..").resolve()
sys.path.insert(0, str(REPO))
from gen_sample.modules.sequence_redundancy import (
    filter_peptide_redundancy, filter_receptor_redundancy)
from gen_sample.modules.sampling import kennard_stone_sample
from gen_sample.modules.utils import enrich_rows

# ── Paths ────────────────────────────────────────────────────────────────────
SCRAPE       = REPO / "scrape_pdb_bind"
SUMMARY_TSV  = SCRAPE / "outputs" / "summary.tsv"
OUT_DIR      = Path("..") / "outputs"
PLOT_DIR     = Path("..") / "plots"
OUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

# ── Configuration ────────────────────────────────────────────────────────────
N_SEEDS          = 25
PEP_THRESH       = 0.70   # gen_sample strict preset
REC_THRESH       = 0.40   # gen_sample strict preset (CD-HIT minimum)

FEATURES = [
    "pep_len", "rec_len",
    "peptide_pct_helix", "peptide_pct_beta", "peptide_pct_coil",
    "receptor_pct_helix", "receptor_pct_beta", "receptor_pct_coil",
    "n_contacts", "n_hbonds",
    "binding_kd", "resolution_f",
]
TARGET = "dG_kcal_mol"

# gen_sample dim names for the 12 features (used by KS sampler)
DIMS_2D  = ["dg", "pep_len"]
DIMS_4D  = ["dg", "pep_len", "rec_len", "n_contacts"]
DIMS_8D  = ["dg", "pep_len", "rec_len", "n_contacts", "n_hbonds",
            "pep_helix", "pep_beta", "pep_coil"]
DIMS_12D = ["pep_len", "rec_len",
            "pep_helix", "pep_beta", "pep_coil",
            "rec_helix", "rec_beta", "rec_coil",
            "n_contacts", "n_hbonds",
            "binding_kd", "resolution"]

STRAT_META = [
    ("Random",                "#888888"),
    ("Seq-70%\nKS 2D",       "#9ecae1"),
    ("Seq-70%\nKS 4D",       "#4292c6"),
    ("Seq-70%\nKS 8D",       "#084594"),
    ("Seq-70%\nKS 12D",      "#c44e52"),
    ("Seq-70%+Rec-40%\nKS 12D", "#55a868"),
]

print(f"Repository : {REPO}")
print(f"Features   : {len(FEATURES)}")
print(f"Strategies : {len(STRAT_META)}")
print(f"N_SEEDS    : {N_SEEDS}")

## 2. Load data

In [ ]:
summary = pd.read_csv(SUMMARY_TSV, sep="\t")

summary["pep_len"]      = summary["peptide"].str.split(":").str[1].astype(float)
summary["rec_len"]      = summary["receptor"].str.split(":").str[1].astype(float)
summary["resolution_f"] = pd.to_numeric(summary["resolution"], errors="coerce")
summary["binding_kd"]   = (summary["binding_type"] == "Kd").astype(float)
for col in ["peptide_pct_helix","peptide_pct_beta","peptide_pct_coil",
            "receptor_pct_helix","receptor_pct_beta","receptor_pct_coil"]:
    summary[col] = pd.to_numeric(summary[col], errors="coerce").fillna(0.0)
for col in ["n_contacts", "n_hbonds"]:
    summary[col] = pd.to_numeric(summary[col], errors="coerce")

print(f"Loaded {len(summary)} entries")
print(f"\nn_contacts : {summary['n_contacts'].isna().sum()} NA  "
      f"(range {summary['n_contacts'].min():.0f}–{summary['n_contacts'].max():.0f})")
print(f"n_hbonds   : {summary['n_hbonds'].isna().sum()} NA  "
      f"(range {summary['n_hbonds'].min():.0f}–{summary['n_hbonds'].max():.0f})")
print(f"\npeptide_ss_class:")
print(summary["peptide_ss_class"].value_counts().to_string())

## 3. Feature matrix

In [ ]:
df = summary.dropna(subset=["n_contacts", "n_hbonds", TARGET]).copy()
df["n_contacts"]   = df["n_contacts"].astype(float)
df["n_hbonds"]     = df["n_hbonds"].astype(float)
df["resolution_f"] = df["resolution_f"].fillna(df["resolution_f"].median())

print(f"Working dataset : {len(df)} entries")
print(f"ΔG range        : {df[TARGET].min():.2f} to {df[TARGET].max():.2f} kcal/mol")
print(f"n_contacts      : {df['n_contacts'].min():.0f} to {df['n_contacts'].max():.0f}")
print(f"n_hbonds        : {df['n_hbonds'].min():.0f} to {df['n_hbonds'].max():.0f}")
print(f"\npeptide_ss_class:")
print(df["peptide_ss_class"].value_counts().to_string())

## 4. Utility functions

In [ ]:
# ── gen_sample bridge ────────────────────────────────────────────────────────
def _df_to_rows(df):
    """Convert DataFrame to gen_sample-compatible rows.
    Uses median-filled resolution_f so resolution dim works for all entries.
    """
    rows = df.to_dict("records")
    for r in rows:
        rf = r.get("resolution_f")
        if rf is not None and not (isinstance(rf, float) and __import__("math").isnan(rf)):
            r["resolution"] = str(rf)
    return enrich_rows(rows)

def _rows_to_df(rows, ref_df):
    ids = {r["pdb_id"] for r in rows}
    return ref_df[ref_df["pdb_id"].isin(ids)].copy()

def gs_dedup_pep(df, threshold=PEP_THRESH):
    rows = filter_peptide_redundancy(df.to_dict("records"), threshold)
    return _rows_to_df(rows, df)

def gs_dedup_rec(df, threshold=REC_THRESH):
    rows = filter_receptor_redundancy(df.to_dict("records"), threshold)
    return _rows_to_df(rows, df)

def gs_ks(df, n, dims):
    # Kennard-Stone using gen_sample module.
    rows     = _df_to_rows(df)
    selected = kennard_stone_sample(rows, n, dims=dims)
    return _rows_to_df(selected, df)

# ── Metrics ───────────────────────────────────────────────────────────────────
def metrics(y_true, y_pred):
    # Returns (rmse, mae, pearson_r, spearman_rho).
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r    = float(pearsonr(y_true, y_pred)[0])
    rho  = float(spearmanr(y_true, y_pred)[0])
    return rmse, mae, r, rho

def internal_diversity(df_sub):
    X = StandardScaler().fit_transform(df_sub[FEATURES].values.astype(float))
    return cdist(X, X).mean()

print("Utility functions defined.")
print("  gs_dedup_pep   — filter_peptide_redundancy (Biopython NW, BLOSUM62)")
print("  gs_dedup_rec   — filter_receptor_redundancy (CD-HIT)")
print("  gs_ks          — kennard_stone_sample (gen_sample)")

## 5. Scenario A: same-pool curation (80/20 split)

**Design:**
- Apply each strategy to the full dataset → curate N entries
- Split curated set 80/20: train on 80%, test on 20%
- Seeds ×25: KS sets are deterministic; only Random and the 80/20 split vary per seed

**Question:** does curation improve within-set generalisation at a fixed budget of N entries?

In [ ]:
def run_scenario_a(n_total, seeds=N_SEEDS):
    # Scenario A: curate N from full dataset, split 80/20.
    print(f"Scenario A · N={n_total} · computing dedup+KS sets...")
    t0 = time.time()

    # Pre-compute deduped pools and KS sets (deterministic — done once)
    pep_dd     = gs_dedup_pep(df)
    pep_rec_dd = gs_dedup_rec(pep_dd)
    print(f"  pep-dedup: {len(pep_dd)}  pep+rec-dedup: {len(pep_rec_dd)}")

    ks_sets = [
        None,                                   # Random: resampled per seed
        gs_ks(pep_dd, n_total, DIMS_2D),
        gs_ks(pep_dd, n_total, DIMS_4D),
        gs_ks(pep_dd, n_total, DIMS_8D),
        gs_ks(pep_dd, n_total, DIMS_12D),
        gs_ks(pep_rec_dd, n_total, DIMS_12D),
    ]
    print(f"  KS sets ready  ({time.time()-t0:.1f}s)")

    res = {i: [] for i in range(len(STRAT_META))}
    for seed in range(seeds):
        for i in range(len(STRAT_META)):
            curated = (df.sample(n=min(n_total, len(df)), random_state=seed)
                       if i == 0 else ks_sets[i])
            train_s, test_s = train_test_split(curated, test_size=0.20, random_state=seed)
            sc = StandardScaler()
            rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                       random_state=42, n_jobs=-1)
            rf.fit(sc.fit_transform(train_s[FEATURES].values.astype(float)),
                   train_s[TARGET].values)
            yp = rf.predict(sc.transform(test_s[FEATURES].values.astype(float)))
            rm, ma, r, rho = metrics(test_s[TARGET].values, yp)
            res[i].append({"rmse":rm,"mae":ma,"r":r,"rho":rho,
                           "n_train":len(train_s),"n_test":len(test_s)})
    print(f"  Seeds done  ({time.time()-t0:.1f}s total)")

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n{'Strategy':<26}  {'RMSE':>12}  {'MAE':>12}  {'r':>10}  {'ρ':>10}  {'n_tr':>5}")
    print("-" * 80)
    for i, (label, _) in enumerate(STRAT_META):
        rmses=[r["rmse"] for r in res[i]]; maes=[r["mae"] for r in res[i]]
        rs=[r["r"] for r in res[i]]; rhos=[r["rho"] for r in res[i]]
        nt=int(np.mean([r["n_train"] for r in res[i]]))
        print(f"{label.replace(chr(10),' '):26s}  "
              f"{np.mean(rmses):5.3f}±{np.std(rmses):.3f}  "
              f"{np.mean(maes):5.3f}±{np.std(maes):.3f}  "
              f"{np.mean(rs):4.2f}±{np.std(rs):.2f}  "
              f"{np.mean(rhos):4.2f}±{np.std(rhos):.2f}  {nt:5d}")

    # ── PCA background ────────────────────────────────────────────────────────
    sc_pca = StandardScaler()
    Xall   = sc_pca.fit_transform(df[FEATURES].values.astype(float))
    pca_   = PCA(n_components=2, random_state=42)
    Xall_2d = pca_.fit_transform(Xall)
    v1, v2  = pca_.explained_variance_ratio_[:2] * 100

    # ── Figure (seed=0 for visualization) ────────────────────────────────────
    NS  = len(STRAT_META)
    fig, axes = plt.subplots(2, NS, figsize=(3.2*NS, 7.5))
    fig.suptitle(
        f"Scenario A · Same-pool curation · N_TOTAL={n_total}\n"
        f"Full dataset ({len(df)}) → curate → 80/20 split · {seeds} seeds · seed=0 shown",
        fontsize=10, fontweight="bold", y=1.01)

    for col, (label, color) in enumerate(STRAT_META):
        curated_v = (df.sample(n=min(n_total,len(df)), random_state=0)
                     if col == 0 else ks_sets[col])
        train_v, test_v = train_test_split(curated_v, test_size=0.20, random_state=0)
        sc_c = StandardScaler()
        rf_c = RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                     random_state=42, n_jobs=-1)
        rf_c.fit(sc_c.fit_transform(train_v[FEATURES].values.astype(float)),
                 train_v[TARGET].values)
        yp_v  = rf_c.predict(sc_c.transform(test_v[FEATURES].values.astype(float)))
        y_tv  = test_v[TARGET].values
        lim   = [df[TARGET].min()-0.3, df[TARGET].max()+0.3]

        rm_m = np.mean([r["rmse"] for r in res[col]]); rm_s = np.std([r["rmse"] for r in res[col]])
        ma_m = np.mean([r["mae"]  for r in res[col]]); ma_s = np.std([r["mae"]  for r in res[col]])
        r_m  = np.mean([r["r"]    for r in res[col]]); r_s  = np.std([r["r"]    for r in res[col]])
        rh_m = np.mean([r["rho"]  for r in res[col]]); rh_s = np.std([r["rho"]  for r in res[col]])

        ax = axes[0, col]
        ax.scatter(y_tv, yp_v, color=color, s=18, alpha=0.65, edgecolors="none")
        ax.plot(lim, lim, "k--", lw=0.7, alpha=0.35)
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal"); ax.tick_params(labelsize=7)
        ax.set_title(
            f"{label}  (tr={len(train_v)}/te={len(test_v)})\n"
            f"RMSE={rm_m:.2f}±{rm_s:.2f}  MAE={ma_m:.2f}±{ma_s:.2f}\n"
            f"r={r_m:.2f}±{r_s:.2f}  ρ={rh_m:.2f}±{rh_s:.2f}", fontsize=7.5)
        if col == 0:
            ax.set_xlabel("Exp ΔG (kcal/mol)", fontsize=8)
            ax.set_ylabel("Pred ΔG (kcal/mol)", fontsize=8)

        ax = axes[1, col]
        Xc_2d  = pca_.transform(sc_pca.transform(curated_v[FEATURES].values.astype(float)))
        Xte_2d = pca_.transform(sc_pca.transform(test_v[FEATURES].values.astype(float)))
        ax.scatter(Xall_2d[:,0], Xall_2d[:,1], c="lightgray", s=8, alpha=0.3,
                   zorder=1, label="Full dataset")
        ax.scatter(Xc_2d[:,0], Xc_2d[:,1], c=color, s=18, alpha=0.45,
                   zorder=2, label=f"Curated N={len(curated_v)}")
        ax.scatter(Xte_2d[:,0], Xte_2d[:,1], c=color, s=38, alpha=0.9, zorder=3,
                   edgecolors="black", linewidths=0.5, label="Test 20%")
        ax.set_title(f"diversity={internal_diversity(curated_v):.2f}", fontsize=8)
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.set_xlabel(f"PC1 ({v1:.0f}%)", fontsize=8)
            ax.set_ylabel(f"PC2 ({v2:.0f}%)", fontsize=8)
        if col == NS-1:
            ax.legend(fontsize=6, loc="upper right")

    plt.tight_layout()
    fname = PLOT_DIR / f"sA_same_pool_N{n_total}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved {fname.name}")

### N = 300

In [ ]:
_ = run_scenario_a(300)

### N = 150

In [ ]:
_ = run_scenario_a(150)

### N = 75

In [ ]:
_ = run_scenario_a(75)

## 6. Scenario B: partial-helix held out (fixed test)

**Design:**
- Fixed test: all entries with `peptide_ss_class = "partial-helix"` (n≈96)
- Train pool: all remaining entries (~643)
- Dedup applied once to train pool; per seed, 80% resample → KS or random selects N entries
- Seeds ×25: KS varies because the 80% subsample varies per seed

**Question:** does curation help when test entries have a distinct secondary structure class?

In [ ]:
test_ph    = df[df["peptide_ss_class"] == "partial-helix"].copy()
train_pool = df[df["peptide_ss_class"] != "partial-helix"].copy()
print(f"Fixed test  (partial-helix) : {len(test_ph)} entries")
print(f"Train pool  (rest)          : {len(train_pool)} entries")
print(f"\npeptide_ss_class in train pool:")
print(train_pool["peptide_ss_class"].value_counts().to_string())

In [ ]:
def run_scenario_b(n_train, seeds=N_SEEDS):
    # Scenario B: partial-helix fixed test.
    print(f"Scenario B · N_TRAIN={n_train} · computing dedup sets...")
    t0 = time.time()

    # Pre-compute deduped pools once (deterministic)
    pep_dd     = gs_dedup_pep(train_pool)
    pep_rec_dd = gs_dedup_rec(pep_dd)
    print(f"  pep-dedup: {len(pep_dd)}  pep+rec-dedup: {len(pep_rec_dd)}  ({time.time()-t0:.1f}s)")

    y_true = test_ph[TARGET].values
    res = {i: [] for i in range(len(STRAT_META))}

    for seed in range(seeds):
        # 80% resample per seed — gives variance to all strategies including KS
        sub_pep     = pep_dd.sample(frac=0.80, random_state=seed)
        sub_pep_rec = pep_rec_dd.sample(frac=0.80, random_state=seed)
        train_sets = [
            train_pool.sample(n=min(n_train, len(train_pool)), random_state=seed),
            gs_ks(sub_pep, n_train, DIMS_2D),
            gs_ks(sub_pep, n_train, DIMS_4D),
            gs_ks(sub_pep, n_train, DIMS_8D),
            gs_ks(sub_pep, n_train, DIMS_12D),
            gs_ks(sub_pep_rec, n_train, DIMS_12D),
        ]
        for i, ts in enumerate(train_sets):
            sc = StandardScaler()
            rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                       random_state=42, n_jobs=-1)
            rf.fit(sc.fit_transform(ts[FEATURES].values.astype(float)), ts[TARGET].values)
            yp = rf.predict(sc.transform(test_ph[FEATURES].values.astype(float)))
            rm, ma, r, rho = metrics(y_true, yp)
            res[i].append({"rmse":rm,"mae":ma,"r":r,"rho":rho,"n":len(ts)})
    print(f"  Seeds done  ({time.time()-t0:.1f}s total)")

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n{'Strategy':<26}  {'RMSE':>12}  {'MAE':>12}  {'r':>10}  {'ρ':>10}  {'n_tr':>5}")
    print("-" * 80)
    for i, (label, _) in enumerate(STRAT_META):
        rmses=[r["rmse"] for r in res[i]]; maes=[r["mae"] for r in res[i]]
        rs=[r["r"] for r in res[i]]; rhos=[r["rho"] for r in res[i]]
        nt=int(np.mean([r["n"] for r in res[i]]))
        print(f"{label.replace(chr(10),' '):26s}  "
              f"{np.mean(rmses):5.3f}±{np.std(rmses):.3f}  "
              f"{np.mean(maes):5.3f}±{np.std(maes):.3f}  "
              f"{np.mean(rs):4.2f}±{np.std(rs):.2f}  "
              f"{np.mean(rhos):4.2f}±{np.std(rhos):.2f}  {nt:5d}")

    # ── PCA background: full train pool ───────────────────────────────────────
    sc_pca = StandardScaler()
    Xp     = sc_pca.fit_transform(train_pool[FEATURES].values.astype(float))
    pca_   = PCA(n_components=2, random_state=42)
    Xp_2d  = pca_.fit_transform(Xp)
    v1, v2 = pca_.explained_variance_ratio_[:2] * 100

    # ── Visualization (seed=0) ────────────────────────────────────────────────
    sub_pep_v     = pep_dd.sample(frac=0.80, random_state=0)
    sub_pep_rec_v = pep_rec_dd.sample(frac=0.80, random_state=0)
    vis_sets = [
        train_pool.sample(n=min(n_train, len(train_pool)), random_state=0),
        gs_ks(sub_pep_v, n_train, DIMS_2D),
        gs_ks(sub_pep_v, n_train, DIMS_4D),
        gs_ks(sub_pep_v, n_train, DIMS_8D),
        gs_ks(sub_pep_v, n_train, DIMS_12D),
        gs_ks(sub_pep_rec_v, n_train, DIMS_12D),
    ]
    lim = [df[TARGET].min()-0.3, df[TARGET].max()+0.3]
    NS  = len(STRAT_META)

    fig, axes = plt.subplots(2, NS, figsize=(3.2*NS, 7.5))
    fig.suptitle(
        f"Scenario B · Partial-helix held out (n={len(test_ph)}) · N_TRAIN={n_train}\n"
        f"Train pool ({len(train_pool)}) · {seeds} seeds (80% resample) · seed=0 shown",
        fontsize=10, fontweight="bold", y=1.01)

    for col, ((label, color), ts_v) in enumerate(zip(STRAT_META, vis_sets)):
        sc_c = StandardScaler()
        rf_c = RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                     random_state=42, n_jobs=-1)
        rf_c.fit(sc_c.fit_transform(ts_v[FEATURES].values.astype(float)),
                 ts_v[TARGET].values)
        yp_v = rf_c.predict(sc_c.transform(test_ph[FEATURES].values.astype(float)))

        rm_m = np.mean([r["rmse"] for r in res[col]]); rm_s = np.std([r["rmse"] for r in res[col]])
        ma_m = np.mean([r["mae"]  for r in res[col]]); ma_s = np.std([r["mae"]  for r in res[col]])
        r_m  = np.mean([r["r"]    for r in res[col]]); r_s  = np.std([r["r"]    for r in res[col]])
        rh_m = np.mean([r["rho"]  for r in res[col]]); rh_s = np.std([r["rho"]  for r in res[col]])

        ax = axes[0, col]
        ax.scatter(y_true, yp_v, color=color, s=18, alpha=0.65, edgecolors="none")
        ax.plot(lim, lim, "k--", lw=0.7, alpha=0.35)
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal"); ax.tick_params(labelsize=7)
        ax.set_title(
            f"{label}  (n={len(ts_v)})\n"
            f"RMSE={rm_m:.2f}±{rm_s:.2f}  MAE={ma_m:.2f}±{ma_s:.2f}\n"
            f"r={r_m:.2f}±{r_s:.2f}  ρ={rh_m:.2f}±{rh_s:.2f}", fontsize=7.5)
        if col == 0:
            ax.set_xlabel("Exp ΔG (kcal/mol)", fontsize=8)
            ax.set_ylabel("Pred ΔG (kcal/mol)", fontsize=8)

        ax = axes[1, col]
        Xts_2d = pca_.transform(sc_pca.transform(ts_v[FEATURES].values.astype(float)))
        ax.scatter(Xp_2d[:,0], Xp_2d[:,1], c="lightgray", s=10, alpha=0.4,
                   zorder=1, label="Train pool")
        ax.scatter(Xts_2d[:,0], Xts_2d[:,1], c=color, s=28, alpha=0.85,
                   zorder=2, label=f"Selected n={len(ts_v)}")
        ax.set_title(f"train diversity={internal_diversity(ts_v):.2f}", fontsize=8)
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.set_xlabel(f"PC1 ({v1:.0f}%)", fontsize=8)
            ax.set_ylabel(f"PC2 ({v2:.0f}%)", fontsize=8)
        if col == NS-1:
            ax.legend(fontsize=6, loc="upper right")

    plt.tight_layout()
    fname = PLOT_DIR / f"sB_partial_helix_N{n_train}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved {fname.name}")

### N = 300

In [ ]:
_ = run_scenario_b(300)

### N = 150

In [ ]:
_ = run_scenario_b(150)

### N = 75

In [ ]:
_ = run_scenario_b(75)

## 7. Save results

In [ ]:
df.to_csv(OUT_DIR / "working_dataset.tsv", sep="\t", index=False)
test_ph.to_csv(OUT_DIR / "test_partial_helix.tsv", sep="\t", index=False)

print("Outputs:")
for f in sorted(OUT_DIR.iterdir()): print(f"  outputs/{f.name}")
print("Plots:")
for f in sorted(PLOT_DIR.iterdir()): print(f"  plots/{f.name}")